# 05. Rotation and Incoherence PTQ: QuaRot, SpinQuant, and QuIP#

- **Category:** 14_quantisation / 03_llm_ptq_algorithms
- **Lineage & Citations:** Ashkboos et al. (2024, *QuaRot: Outlier-Free 4-Bit LLMs via Randomized Hadamard Rotations*); Liu et al. (2024, *SpinQuant: LLM Quantization with Learned Rotation Matrices*); Tseng et al. (2024, *QuIP#: Even Better LLM Quantization with Hadamard Incoherence and E8 Lattice*).
- **Purpose:** Eliminate activation outliers at the source by applying orthogonal rotations $\mathbf{Q} \in O(d)$ to activations and weights ($\mathbf{Y} = (\mathbf{X}\mathbf{Q})(\mathbf{Q}^T\mathbf{W})$), dispersing outlier energy uniformly across coordinates and unlocking 4-bit weights and 4-bit activations (W4A4).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [04. LLM.int8() Mixed Precision Outlier Decomposition](04_llm_int8_mixed_precision_outlier_decomposition.ipynb) $\to$ `🔗 Downstream Syntheses:` [06. OmniQuant and HQQ Optimization-Based PTQ](06_omniquant_and_hqq_optimization_based_ptq.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use dense unstructured rotation matrices without fast butterfly structures (dense matrix multiply adds $O(d^2)$ inference overhead; use FWHT or fold rotations into preceding weights); do NOT apply rotations without matching sign flips $\text{diag}(\mathbf{s})$ (un-randomized Hadamard allows structured worst-case cancellations).

## Mathematical Foundations & Orthogonal Incoherence Rotations

1. **Invariant Orthogonal Transformation**:
   $$\mathbf{Y} = \mathbf{X}\mathbf{W} = (\mathbf{X}\mathbf{Q})(\mathbf{Q}^T\mathbf{W}) = \widetilde{\mathbf{X}}\widetilde{\mathbf{W}}, \quad \mathbf{Q}\mathbf{Q}^T = \mathbf{I}$$
2. **Incoherence & Peak Suppression**:
   $$\|\mathbf{x}\mathbf{Q}\|_\infty \le O\left( \sqrt{\frac{\log d}{d}} \right) \|\mathbf{x}\|_2 \implies \text{outliers dispersed into uniform Gaussian-like features}$$
3. **Fast Walsh-Hadamard Transform (FWHT)**:
   $$\mathbf{Q} = \frac{1}{\sqrt{d}}\mathbf{H}_d \text{diag}(\mathbf{s}), \quad \mathbf{H}_{2k} = \frac{1}{\sqrt{2}}\begin{bmatrix} \mathbf{H}_k & \mathbf{H}_k \\ \mathbf{H}_k & -\mathbf{H}_k \end{bmatrix} \implies O(d \log d) \text{ compute}$$
4. **Cayley Stiefel Rotations (SpinQuant)**:
   $$\mathbf{R}(\mathbf{A}) = (\mathbf{I} - \mathbf{A})(\mathbf{I} + \mathbf{A})^{-1}, \quad \mathbf{A} = -\mathbf{A}^T \implies \mathbf{R} \in SO(d)$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/03_llm_ptq_algorithms/05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def construct_hadamard_matrix(dim: int) -> torch.Tensor:
    if dim == 1: return torch.tensor([[1.0]], dtype=torch.float32)
    h_sub = construct_hadamard_matrix(dim // 2)
    factor = 1.0 / (2.0 ** 0.5)
    return factor * torch.cat([torch.cat([h_sub, h_sub], dim=1), torch.cat([h_sub, -h_sub], dim=1)], dim=0)

def construct_randomized_hadamard(dim: int, signs: torch.Tensor) -> torch.Tensor:
    return construct_hadamard_matrix(dim) * signs.unsqueeze(0)

def cayley_orthogonal_rotation(skew_matrix: torch.Tensor) -> torch.Tensor:
    dim = skew_matrix.shape[0]
    eye = torch.eye(dim, dtype=skew_matrix.dtype, device=skew_matrix.device)
    a = 0.5 * (skew_matrix - skew_matrix.T)
    return torch.inverse(eye + a) @ (eye - a)

def quantize_per_token_int4(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    scale = torch.max(torch.abs(x), dim=-1, keepdim=True).values / 7.0
    scale = torch.where(scale == 0.0, torch.ones_like(scale), scale)
    return torch.clamp(torch.round(x / scale), -8.0, 7.0), scale

def quantize_per_column_int4(w: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    scale = torch.max(torch.abs(w), dim=0, keepdim=True).values / 7.0
    scale = torch.where(scale == 0.0, torch.ones_like(scale), scale)
    return torch.clamp(torch.round(w / scale), -8.0, 7.0), scale

def w4a4_gemm(x: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
    q_x, s_x = quantize_per_token_int4(x)
    q_w, s_w = quantize_per_column_int4(weight)
    return (q_x * s_x) @ (q_w * s_w)

In [ ]:
# 1. Linear anchor parity
y = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-5

# 2. Hadamard matrix parity
h16 = construct_hadamard_matrix(16)
assert float(torch.max(torch.abs(h16 - weights["hadamard_16"]))) < 1e-5
q_had = construct_randomized_hadamard(16, weights["signs"])
assert float(torch.max(torch.abs(q_had - weights["q_hadamard"]))) < 1e-5

# 3. Rotated activations and weights parity
x_rot = weights["x_calib"] @ weights["q_hadamard"]
w_rot = weights["q_hadamard"].T @ weights["w_orig"]
assert float(torch.max(torch.abs(x_rot - weights["x_rot"]))) < 1e-5
assert float(torch.max(torch.abs(w_rot - weights["w_rot"]))) < 1e-5

# 4. W4A4 output parity
out_unrot = w4a4_gemm(weights["x_calib"], weights["w_orig"]) + weights["b_orig"]
out_quarot = w4a4_gemm(x_rot, w_rot) + weights["b_orig"]
assert float(torch.max(torch.abs(out_unrot - weights["out_unrotated_w4a4"]))) < 1e-5
assert float(torch.max(torch.abs(out_quarot - weights["out_quarot_w4a4"]))) < 1e-5
print("All QuaRot and randomized Hadamard parity assertions verified!")

In [ ]:
# Comparative SQNR Benchmark: Unrotated W4A4 vs QuaRot Rotated W4A4
y_fp32 = weights["out_fp32"]
sig_pwr = float(torch.mean(y_fp32 ** 2))

def eval_sqnr(y_pred: torch.Tensor) -> float:
    mse = float(torch.mean((y_fp32 - y_pred) ** 2))
    return 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

sqnr_unrot = eval_sqnr(weights["out_unrotated_w4a4"])
sqnr_quarot = eval_sqnr(weights["out_quarot_w4a4"])

print(f"Unrotated W4A4 SQNR: {sqnr_unrot:.2f} dB (decimated by outlier clipping)")
print(f"QuaRot W4A4 SQNR:    {sqnr_quarot:.2f} dB (gain: +{sqnr_quarot - sqnr_unrot:.2f} dB)")
assert sqnr_quarot > sqnr_unrot + 3.0, "QuaRot orthogonal rotation must yield >3 dB SQNR gain over unrotated W4A4!"

In [ ]:
# Latency Microbenchmark: In-place Fast Walsh-Hadamard Transform (FWHT)
b, dim = 64, 4096
x_mock = torch.randn(b, dim)
inv_sqrt2 = 1.0 / (2.0 ** 0.5)

t0 = time.perf_counter()
for _ in range(50):
    # Butterfly FWHT execution simulation
    h = 1
    while h < dim:
        step = h * 2
        x_mock = x_mock.view(-1, step)
        u, v = x_mock[:, :h].clone(), x_mock[:, h:].clone()
        x_mock[:, :h] = (u + v) * inv_sqrt2
        x_mock[:, h:] = (u - v) * inv_sqrt2
        h = step
    x_mock = x_mock.view(b, dim)
t_fwht = (time.perf_counter() - t0) / 50 * 1e3

print(f"Fast Walsh-Hadamard Transform (64x4096) latency: {t_fwht:.3f} ms")
assert t_fwht < 25.0, "FWHT must complete in <25 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Outliers are Basis Artifacts**: Pre-training coordinates create systemic outliers; rotating by orthogonal matrices $\mathbf{Q}$ disperses outlier magnitude across all coordinates.
2. **FWHT $O(d \log d)$ Incoherence**: Randomized Walsh-Hadamard transforms eliminate outliers in $O(d \log d)$ time without matrix multiplications.
3. **True W4A4 Feasibility**: Dispersing outliers unlocks symmetric 4-bit activations and 4-bit weights without mixed-precision FP16 tables.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.md](../../../markdown/14_quantisation/03_llm_ptq_algorithms/05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.md)
- **PyTorch 2.x Module:** [05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.py](../../../pytorch2/14_quantisation/03_llm_ptq_algorithms/05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.py)
- **Functional JAX Module:** [05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.py](../../../jax/14_quantisation/03_llm_ptq_algorithms/05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.py)
- **Native Rust Candle Track:** [05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.rs](../../../candle/14_quantisation/03_llm_ptq_algorithms/05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.rs)
- **Hardware Kernel (CPU / WGSL):** [05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.rs](../../../kernels/14_quantisation/03_llm_ptq_algorithms/05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.rs)
- **Golden SafeTensors:** [05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.safetensors](../../../weights/14_quantisation/03_llm_ptq_algorithms/05_rotation_and_incoherence_ptq_quarot_spinquant_and_quip_sharp.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. LLM.int8() Mixed Precision Outlier Decomposition](04_llm_int8_mixed_precision_outlier_decomposition.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 06. OmniQuant and HQQ Optimization-Based PTQ](06_omniquant_and_hqq_optimization_based_ptq.ipynb) &rarr;